# Week 55, Day 1: Least squares as maximum likelihood, and how sure are the weights?

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w055/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Standard errors and their coverage

Write standard_errors(X, y), returning the least-squares weights and their standard errors (using the unbiased noise estimate). Then simulate 2,000 datasets with the same design, and compare the average reported SE with the actual spread of the estimates, and the coverage of the 95% intervals.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
r = np.random.default_rng(551)
n = 40
X = np.c_[np.ones(n), r.normal(size=(n, 2))]
w_true, sigma = np.array([1.0, 0.5, -0.3]), 1.0
y = X @ w_true + sigma * r.normal(size=n)

def standard_errors(X, y):
    w = np.linalg.solve(X.T @ X, X.T @ y)
    res = y - X @ w
    s2 = ...
    se = ...
    return w, se

t975 = stats.t.ppf(0.975, n - 3)
ests, ses, covered = [], [], []
for rep in range(2000):
    yy = X @ w_true + sigma * r.normal(size=n)
    w_, se_ = standard_errors(X, yy)
    ests.append(w_); ses.append(se_); covered.append(np.abs(w_ - w_true) <= t975 * se_)
ests, ses = np.array(ests), np.array(ses)
sd_est, mean_se, coverage = ests.std(0), ses.mean(0), np.mean(covered, axis=0)
print("actual SD of estimates:", np.round(sd_est, 3))
print("average reported SE:   ", np.round(mean_se, 3))
print("coverage of 95% CIs:   ", np.round(coverage, 3))

In [ ]:
import numpy as np
def test_se():
    "standard_errors returns the square roots of the diagonal of sigma2_hat (X'X)^-1, with sigma2_hat = RSS / (n - p)"
    w_, se_ = standard_errors(X, y)
    res = y - X @ w_; s2 = res @ res / (n - X.shape[1])
    assert np.allclose(se_, np.sqrt(np.diag(s2 * np.linalg.inv(X.T @ X))))
def test_match():
    "the average reported SE matches the spread of the estimates over 2,000 datasets within 5%"
    assert np.all(np.abs(mean_se / sd_est - 1) < 0.05)
def test_coverage():
    "the 95% intervals cover the true weights about 95% of the time"
    assert np.all(np.abs(coverage - 0.95) < 0.02)

test_se()
test_match()
test_coverage()
print("All checks passed")

### Exercise: How correlation inflates uncertainty

Write vif(Z, j). Then for each correlation in rhos, simulate 500 datasets of 100 trials with two regressors correlated at rho, record the spread of the first weight's estimate, and divide it by the spread at rho = 0.

In [ ]:
def vif(Z, j):
    others = np.c_[np.ones(len(Z)), np.delete(Z, j, axis=1)]
    beta, *_ = np.linalg.lstsq(others, Z[:, j], rcond=None)
    resid = Z[:, j] - others @ beta
    r2 = ...
    return 1 / (1 - r2)

rhos = [0.0, 0.5, 0.8, 0.9, 0.95]
spread = []
for rho in rhos:
    est = []
    for rep in range(500):
        z1, z2 = r.normal(size=100), r.normal(size=100)
        x1, x2 = z1, rho * z1 + np.sqrt(1 - rho ** 2) * z2
        yy = 1.0 * x1 + 0.5 * x2 + r.normal(size=100)
        est.append(np.linalg.lstsq(np.c_[np.ones(100), x1, x2], yy, rcond=None)[0][1])
    spread.append(np.std(est))
se_ratio = np.array(spread) / spread[0]
Z = np.c_[x1, x2]
for rho, s in zip(rhos, se_ratio):
    print(f"rho = {rho:.2f}: SE of w1 is {s:.2f} times larger (sqrt of VIF: {np.sqrt(1 / (1 - rho ** 2)):.2f})")
print(f"VIF of the last simulated pair: {vif(Z, 0):.1f}")

In [ ]:
import numpy as np
def test_vif():
    "vif(Z, j) regresses column j on the others (with intercept) and returns 1 / (1 - R²)"
    Z = np.c_[r.normal(size=500), r.normal(size=500)]
    assert abs(vif(Z, 0) - 1) < 0.05
def test_ratio():
    "the SE of a weight grows by about the square root of the VIF as the correlation rises"
    assert abs(se_ratio[-1] - np.sqrt(1 / (1 - 0.95 ** 2))) / np.sqrt(1 / (1 - 0.95 ** 2)) < 0.15

test_vif()
test_ratio()
print("All checks passed")

### Exercise: Where predictions are safe

Write pred_sd(X, x0), the standard deviation of the prediction at x0 when sigma = 1. Compare a typical input (1, 1, 1: both regressors high together) with an atypical one (1, 1, -1: one high, one low) for the design with correlation 0.95.

In [ ]:
z1, z2 = r.normal(size=100), r.normal(size=100)
Xc = np.c_[np.ones(100), z1, 0.95 * z1 + np.sqrt(1 - 0.95 ** 2) * z2]

def pred_sd(X, x0):
    return ...

sd_typical = pred_sd(Xc, np.array([1.0, 1.0, 1.0]))
sd_atypical = pred_sd(Xc, np.array([1.0, 1.0, -1.0]))
se_w = np.sqrt(np.diag(np.linalg.inv(Xc.T @ Xc)))
print(f"SE of each weight: {np.round(se_w[1:], 3)}")
print(f"prediction SD, regressors agreeing: {sd_typical:.3f}; disagreeing: {sd_atypical:.3f}")

In [ ]:
import numpy as np
def test_formula():
    "pred_sd returns sigma times the square root of x0' (X'X)^-1 x0"
    x0 = np.array([1.0, 1.0, 1.0])
    assert abs(pred_sd(Xc, x0) - np.sqrt(x0 @ np.linalg.inv(Xc.T @ Xc) @ x0)) < 1e-12
def test_compare():
    "the atypical input's prediction is at least 3 times more uncertain than the typical one's"
    assert sd_atypical > 3 * sd_typical

test_formula()
test_compare()
print("All checks passed")